In [1]:
%load_ext autoreload
%autoreload 2

# **LSTM *inter-epoch***

Este es el primer modelo secuencial del proyecto. Complementa al baseline tabular (XGBoost), que clasifica cada época de forma aislada: acá el modelo aprende explícitamente la dinámica temporal entre épocas. El sueño tiene estructura (las etapas siguen patrones tipo Wake $\to$ N1 $\to$ N2 $\to$ N3 $\to \dots \to$ REM, con transiciones que no son aleatorias), y una red recurrente puede explotar ese contexto.

El modelo es una LSTM bidireccional (BiLSTM) many-to-many: procesa una noche completa como una secuencia y emite una predicción por época. 

## **Arquitectura**
$$
\text{features}[T, F]  \to  \text{encoder}  \to  \texttt{LSTM} \ \text{(2 capas, bidireccional)}  \to  \text{Dropout} \to \text{Linear}  \to  \text{logits}[T, 5]
$$

- ***encoder***: en esta primer versión es IdentityEncoder (deja pasar las features pre-computadas tal cual). Es un punto de extensión deliberado: a futuro se reemplaza por una CNN 1D sobre la señal cruda para armar el híbrido $\texttt{CNN1D}\to\texttt{LSTM}$, sin tocar el resto del pipeline.
- ***LSTM***: $\texttt{hidden\_size=128, num\_layers=2, dropout=0.3, bidirectional=True}$. "Bidireccional" = lee la noche de adelante hacia atrás y de atrás hacia adelante, así cada época ve contexto pasado y futuro (en sleep staging offline tenemos la noche entera, no hay restricción de causalidad). El flag `bidirectional` permite pasar de LSTM a BiLSTM sin reescribir nada.
- ***Cabeza lineal***: proyecta el estado oculto de cada timestep a 5 logits, uno por etapa.
- ***Input***: una secuencia por noche, $\text{features}[T, F]$, donde $T \approx 800–900$ épocas (longitud variable por noche) y $F = 122$ features por época.
- ***Output***: $\text{logits}[T, 5]$ para cada época, un puntaje por clase. La predicción es el argmax sobre las 5 clases.
- ***Clases*** (target = *expert_label*): {0:Wake, 1:N1, 2:N2, 3:N3, 4:REM}. La etiqueta 5:Unknown no es una clase a predecir: se ignora.

## Uso de **Features Pre-computadas**

El modelo utiliza `epoch_features.csv` (ejecutar [notebooks/baseline.ipynb](baseline.ipynb)). El CSV tiene una fila por época, con columnas subject, night, epoch, <122 features>, label, dreem. El modelo:

1. Agrupa por (subject, night) $\to$ cada noche es una secuencia. Las filas se ordenan por epoch para respetar el orden temporal (`NightSequenceDataset`).
2. Usa como features todas las columnas que no son metadata (META_COLS = subject, night, epoch, label, dreem). Son las mismas features intra/inter-época que usa el baseline (HRV, acelerometría, lags/leads, medias móviles, etc.).
3. Usa label (experto) como target y conserva dreem solo como referencia (no entra al modelo).

A diferencia del baseline, que trata cada fila como un ejemplo independiente, acá las filas de una misma noche forman una sola secuencia ordenada que la LSTM recorre época por época.

## Armado y Procesamiento de **Secuencias**

**Split por sujeto**: Train/val/test se separan con sujetos disjuntos (`split_subjects`), no por noche ni por época. Esto
evita data leakage: dos noches del mismo paciente comparten fisiología, y si cayeran en train y test el modelo "haría
trampa". 

**Normalización (estandarización)**: Se calcula media y desvío de cada feature solo sobre el train (`feature_stats`,
ignorando NaN) y se aplica $(x − \mu) / \sigma$ a los tres splits. Las stats salen siempre del train para no filtrar
información de val/test.

**NaN de borde**: Las features inter-época (_lag, _lead, etc.) son NaN en los bordes de cada noche (no hay vecino). Se imputan a 0 (= la media, ya post-estandarización) con `nan_to_num`.

**Padding + packing**: Las noches tienen longitudes distintas, pero un batch necesita un tensor rectangular. En cada batch (collate_nights):
- Se paddea cada noche hasta la más larga del batch: las features con 0.0 y las labels con UNKNOWN (5).
- Se guarda un vector lengths con el largo real de cada noche.
- El modelo usa `pack_padded_sequence` / `pad_packed_sequence`: la LSTM ignora los timesteps de padding (no contaminan los estados ocultos ni los gradientes). El padding es solo un relleno para poder batchear, no datos reales.

## **Loss Function**

Se usa CrossEntropyLoss(weight=..., **ignore_index=UNKNOWN**). Tanto las épocas Unknown como el padding (ambos marcados con 5) quedan fuera de la loss, pero se mantienen dentro de la secuencia para no romper la continuidad temporal (sacarlas desalinearía el contexto que ve la recurrente).

## **Desbalance de Clases**

Las etapas están desbalanceadas (N2 abunda, N1 escasea). Se pasan pesos inversos a la frecuencia de cada clase (calculados sobre el train, `class_weights`) a la loss, para que el modelo no colapse a predecir siempre la clase mayoritaria.

## Entrenamiento

- **Optimizador**: Adam, con gradient clipping (grad_clip=5.0) para estabilizar el entrenamiento de la recurrente.
- **Métricas por época** (compute_metrics, calculadas solo sobre épocas válidas, sin padding ni Unknown): *Cohen's Kappa* (métrica principal, corrige el acuerdo por azar), *macro F1* (promedia las 5 clases por igual, robusto al desbalance), *accuracy* y *matriz de confusión*. Además se reporta una versión colapsada a 4 clases (Wake / Light=N1+N2 / Deep=N3 / REM) para comparar con el paper de referencia.
- **Checkpoint**: se guarda el mejor modelo por Kappa de validación (.pt). El archivo incluye los pesos (model_state), la
config, y las stats de normalización (mean/std). Al final, se recarga ese mejor checkpoint y se evalúa en test.
- **Reproducibilidad**: seed fija (set_seed) sobre Python/NumPy/PyTorch, y toda la config centralizada en la dataclass
Config.

In [2]:
import sys
sys.path.append("..")

In [3]:
from src.lstm import ConfigLSTM, train

model, history, test_metrics = train(ConfigLSTM())

sujetos -> train 33 | val 7 | test 7


entrenando: 100%|██████████| 60/60 [06:20<00:00,  6.33s/epoch, loss=0.2611, val_kappa=0.4146, macroF1=0.5277, acc=0.5652, best=0.4499] 



TEST (mejor ckpt, val kappa 0.4499, epoch 27):
  kappa 0.4435 | macroF1 0.5592 | acc 0.5666
  4-clases: kappa 0.4769 | macroF1 0.6448 | acc 0.6405


## Evaluación en test — métricas comparables con XGBoost

Recargamos el **mejor checkpoint** (seleccionado por Kappa de validación) y evaluamos en el
**mismo test set** (mismo split por sujeto y semilla que el entrenamiento). Reportamos las
mismas métricas que `baseline.ipynb`/XGBoost para comparación directa: Cohen's Kappa y
F1-macro vs *Expert* (target) y vs *Dreem* (2da anotación), `classification_report`,
`confusion_matrix` a 5 clases y la versión colapsada a 4 (Wake / Light=N1+N2 / Deep=N3 / REM).

Corresponde al caso del **encoder intra-época identidad** (`IdentityEncoder`): la LSTM consume
las features handcrafted pre-computadas tal cual, sin encoder sobre señal cruda.

In [4]:
import numpy as np
import pandas as pd
import torch
from src.lstm import LSTM, split_subjects, NightSequenceDataset, UNKNOWN

# recarga del mejor checkpoint (incluye config, mean/std de train)
ckpt = torch.load(ConfigLSTM().ckpt_path, map_location='cpu', weights_only=False)
cfg = ckpt['config']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
mean, std = ckpt['mean'], ckpt['std']
print(f"mejor checkpoint -> epoch {ckpt['epoch']}  |  val kappa {ckpt['val_kappa']:.4f}")

model = LSTM(cfg).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# mismo split por sujeto (misma seed/fracciones) -> test set idéntico al del entrenamiento
df = pd.read_csv(cfg.features_path)
splits, subj = split_subjects(df, cfg)
test_ds = NightSequenceDataset(splits['test'], cfg.feature_cols, mean, std)
print('sujetos test:', sorted(subj['test']))

mejor checkpoint -> epoch 27  |  val kappa 0.4499
sujetos test: [np.int64(20), np.int64(31), np.int64(47), np.int64(49), np.int64(52), np.int64(56), np.int64(60)]


In [5]:
# predicciones por noche (batch de 1 -> sin padding), alineadas con expert y dreem
y_true, y_pred, y_dreem = [], [], []
with torch.no_grad():
    for i in range(len(test_ds)):
        feats, labels = test_ds[i]
        _, pos = test_ds.groups[i]
        rows = test_ds.df.iloc[pos]
        lengths = torch.tensor([feats.shape[0]], dtype=torch.long)
        logits = model(feats[None].to(device), lengths)[0]      # [T, 5]
        pred = logits.argmax(-1).cpu().numpy()
        exp = labels.numpy()
        valid = exp != UNKNOWN                                   # descarta Unknown (5)
        y_true.append(exp[valid])
        y_pred.append(pred[valid])
        y_dreem.append(rows['dreem'].to_numpy()[valid])

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)
y_dreem = np.concatenate(y_dreem)
print('épocas de test evaluadas:', len(y_true))

épocas de test evaluadas: 31465


In [6]:
from sklearn.metrics import f1_score, cohen_kappa_score, classification_report, confusion_matrix

# vs Expert (target de entrenamiento)
f1_exp = f1_score(y_true, y_pred, average='macro')
kappa_exp = cohen_kappa_score(y_true, y_pred)

# vs Dreem (2da anotación); descartamos épocas Dreem Unknown
mask = y_dreem != 5
f1_dreem = f1_score(y_dreem[mask], y_pred[mask], average='macro')
kappa_dreem = cohen_kappa_score(y_dreem[mask], y_pred[mask])

print(f"Expert  ->  F1-macro: {f1_exp:.3f}   Cohen's Kappa: {kappa_exp:.3f}")
print(f"Dreem   ->  F1-macro: {f1_dreem:.3f}   Cohen's Kappa: {kappa_dreem:.3f}")
print()
print("Reporte por clase (vs Expert):")
print(classification_report(y_true, y_pred, target_names=['Wake', 'N1', 'N2', 'N3', 'REM']))
print("Matriz de confusión (vs Expert):")
print(confusion_matrix(y_true, y_pred, labels=range(5)))

Expert  ->  F1-macro: 0.559   Cohen's Kappa: 0.443
Dreem   ->  F1-macro: 0.528   Cohen's Kappa: 0.413

Reporte por clase (vs Expert):
              precision    recall  f1-score   support

        Wake       0.75      0.57      0.65      3492
          N1       0.23      0.44      0.31      2940
          N2       0.73      0.45      0.56     12259
          N3       0.69      0.74      0.72      6030
         REM       0.49      0.67      0.57      6744

    accuracy                           0.57     31465
   macro avg       0.58      0.58      0.56     31465
weighted avg       0.63      0.57      0.58     31465

Matriz de confusión (vs Expert):
[[1988  873  173   93  365]
 [ 285 1302  307  116  930]
 [ 246 2018 5506 1587 2902]
 [  51  362  612 4480  525]
 [  81 1038  897  176 4552]]


### Colapso a 4 clases (Wake / Light=N1+N2 / Deep=N3 / REM)

Comparable con SLAMSS-IFS y con la sección de 4 clases del baseline.

In [7]:
map4 = {0: 0, 1: 1, 2: 1, 3: 2, 4: 3}
names4 = ['Wake', 'Light', 'Deep', 'REM']
remap = np.vectorize(map4.get)

y_true4, y_pred4 = remap(y_true), remap(y_pred)
f1_exp4 = f1_score(y_true4, y_pred4, average='macro')
kappa_exp4 = cohen_kappa_score(y_true4, y_pred4)

mask = y_dreem != 5
y_dreem4 = remap(y_dreem[mask])
y_pred4_d = remap(y_pred[mask])
f1_dreem4 = f1_score(y_dreem4, y_pred4_d, average='macro')
kappa_dreem4 = cohen_kappa_score(y_dreem4, y_pred4_d)

print(f"Expert (4 clases)  ->  F1-macro: {f1_exp4:.3f}   Cohen's Kappa: {kappa_exp4:.3f}")
print(f"Dreem  (4 clases)  ->  F1-macro: {f1_dreem4:.3f}   Cohen's Kappa: {kappa_dreem4:.3f}")
print()
print(classification_report(y_true4, y_pred4, target_names=names4))
print(confusion_matrix(y_true4, y_pred4, labels=range(4)))

Expert (4 clases)  ->  F1-macro: 0.645   Cohen's Kappa: 0.477
Dreem  (4 clases)  ->  F1-macro: 0.638   Cohen's Kappa: 0.470

              precision    recall  f1-score   support

        Wake       0.75      0.57      0.65      3492
       Light       0.70      0.60      0.65     15199
        Deep       0.69      0.74      0.72      6030
         REM       0.49      0.67      0.57      6744

    accuracy                           0.64     31465
   macro avg       0.66      0.65      0.64     31465
weighted avg       0.66      0.64      0.64     31465

[[1988 1046   93  365]
 [ 531 9133 1703 3832]
 [  51  974 4480  525]
 [  81 1935  176 4552]]
